# 02 RADAR Velocity

This notebook reconstructs RADAR target velocity using the prepared dataset outputs from `01_dataset_preparation.ipynb`.

The original RADAR velocity analysis logic from `condition_velocity.ipynb` is retained.

The final RADAR velocity table is saved to `results/condition/` for use by later analysis notebooks.

In [1]:
import sys
sys.path.append("../..")

from pathlib import Path

import pandas as pd
from truckscenes import TruckScenes

from src.config import DATA_ROOT, SENSOR_ROOT
from src.radar_velocity import (
    build_target_radar_points_df,
    build_radar_radial_df,
    build_radar_velocity_df,
)

## 1. Load Prepared Data

Load the dataset preparation outputs generated by `01_dataset_preparation.ipynb`.

In [2]:
RESULT_DIR = Path("../../results/condition")

In [3]:
available_sample_data = pd.read_csv(RESULT_DIR / "available_sample_data.csv")
radar_observation_df = pd.read_csv(RESULT_DIR / "radar_observation_df.csv")
vehicle_reference_df = pd.read_csv(RESULT_DIR / "vehicle_reference_df.csv")

In [4]:
trucksc = TruckScenes(dataroot=DATA_ROOT)

Auto-detected latest version: v1.2-trainval
Loading truckscenes tables for version v1.2-trainval...
11 attribute,
18 calibrated_sensor,
27 category,
1200243 ego_motion_cabin,
1200183 ego_motion_chassis,
1201993 ego_pose,
43769 instance,
23902 sample,
819536 sample_annotation,
2586264 sample_data,
598 scene,
18 sensor,
4 visibility,
598 weather_annotation,
Done loading in 41.134 seconds.
Reverse indexing ...
Done reverse indexing in 24.4 seconds.
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 2. Build RADAR Velocity

The RADAR velocity analysis reconstructs target motion from point-level radial-velocity information.

Each RADAR return contributes a radial-velocity constraint along its own line-of-sight (LOS) direction. After ego-motion compensation and coordinate transformation, multiple RADAR constraints are combined to estimate the target's 2D ground-plane velocity.

The reconstruction is performed for each annotation-derived velocity interval using RADAR observations associated with both the start and end annotations. This keeps the RADAR estimate aligned with the same temporal interval used for the annotation-derived reference velocity and provides multiple LOS constraints for the 2D reconstruction.

A RADAR velocity is considered **reconstructable** only when:

1. RADAR constraints are available at both interval endpoints; and
2. the combined 2D constraint matrix has rank 2.

A rank-2 system is required because the target velocity contains two unknown ground-plane components, `vx` and `vy`. Multiple sufficiently independent LOS directions are therefore needed to reconstruct the full 2D velocity. The current implementation explicitly combines the start- and end-point constraints and rejects systems with rank below 2.

After a full-rank solution is obtained, its numerical stability is assessed using the condition number of the 2D constraint matrix. The project baseline uses:

`is_stable_2d = True` when `condition_2d <= 20`.

The threshold of 20 is not an official TruckScenes threshold. It is the existing project baseline established during the earlier RADAR velocity analysis.

A sensitivity check was performed using condition-number thresholds of 5, 10, 20, 30, 50 and 100. Median velocity error remained broadly similar around thresholds of 20–30, whereas looser thresholds increasingly admitted estimates with larger error spread. The existing threshold of 20 was therefore retained as a conservative and methodologically consistent baseline rather than re-tuning the threshold for the current analysis.

The RADAR results are therefore interpreted at two levels:

- **Reconstructable RADAR velocity:** a full-rank 2D velocity solution can be obtained.
- **Stable RADAR velocity:** the reconstructed solution also satisfies the numerical-stability criterion.

All full-rank RADAR estimates are retained when analysing reconstruction availability. Only numerically stable estimates are used for final RADAR velocity-accuracy and paired RADAR–LiDAR accuracy comparisons.

The stability criterion assesses numerical conditioning rather than prediction correctness. A stable solution is not guaranteed to be accurate, while an unstable solution is not necessarily incorrect; however, unstable systems are more sensitive to measurement noise and small changes in the RADAR constraints.

In [5]:
radar_tokens = set(radar_observation_df["Token"])
radar_sample_tokens = set(radar_observation_df["Sample Token"])
radar_sample_data = available_sample_data[available_sample_data["token"].isin(radar_tokens)].copy()
radar_sample_data_lookup = radar_sample_data.set_index("token").to_dict("index")

ego_pose_tokens = set(radar_sample_data["ego_pose_token"].dropna())
radar_ego_pose_lookup = {
    x["token"]: x
    for x in trucksc.ego_pose
    if x["token"] in ego_pose_tokens
}

calibrated_sensor_tokens = set(radar_sample_data["calibrated_sensor_token"].dropna())
radar_calibrated_sensor_lookup = {
    x["token"]: x
    for x in trucksc.calibrated_sensor
    if x["token"] in calibrated_sensor_tokens
}

radar_annotations = [
    x
    for x in trucksc.sample_annotation
    if x["sample_token"] in radar_sample_tokens
]

In [6]:
target_radar_points_df = build_target_radar_points_df(
    radar_observation_df,
    radar_annotations,
    trucksc.instance,
    trucksc.category,
    radar_sample_data_lookup,
    radar_ego_pose_lookup,
    radar_calibrated_sensor_lookup,
    SENSOR_ROOT,
)

radar_radial_df = build_radar_radial_df(
    target_radar_points_df,
    radar_sample_data_lookup,
    radar_ego_pose_lookup,
    radar_calibrated_sensor_lookup,
    trucksc.ego_motion_cabin,
)

radar_velocity_df = build_radar_velocity_df(
    radar_radial_df,
    vehicle_reference_df,
)

In [7]:
print("Reference intervals:", len(vehicle_reference_df))
print("RADAR reconstructable intervals:", len(radar_velocity_df))
print("RADAR stable intervals:", radar_velocity_df["is_stable_2d"].sum())

Reference intervals: 534039
RADAR reconstructable intervals: 97517
RADAR stable intervals: 38179


## 3. Save RADAR Velocity

In [8]:
radar_velocity_df.to_csv(RESULT_DIR / "radar_velocity_df.csv", index=False)
print("RADAR velocity results saved.")

RADAR velocity results saved.
